# CALCULATING SNN FOR ENVI LATENT ONLY AND USING IT TO INFER METACELLS

In [2]:

from __future__ import annotations
from typing import TYPE_CHECKING
from warnings import warn
import logging
import os
import warnings
import datetime

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # FATA

warnings.filterwarnings('ignore')
logging.getLogger('tensorflow').setLevel(logging.FATAL)
logging.getLogger('tensorflow_probability').setLevel(logging.FATAL)

from IPython.display import clear_output


import numpy as np
import pandas as pd
import scanpy as sc
import colorcet
import sklearn.neighbors
import scipy.sparse
import numpy as np
#from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from typing import Optional
from typing import Union
import numpy as np
import pandas as pd
from anndata import AnnData  # type: ignore
from numpy.typing import NDArray
import umap.umap_ as umap
import matplotlib
import matplotlib.pyplot as plt
import colorcet
import sklearn.neighbors
       # Plotting ENVI latent and COVET matrices
import umap
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse

import metacells.parameters as pr
import metacells.utilities as ut
from scipy.sparse import load_npz

In [ ]:
def compute_knn_indices(
    X: np.ndarray, n_neighbors: int
) -> tuple[np.ndarray, np.ndarray]:
    nbrs = NearestNeighbors(n_neighbors=n_neighbors, algorithm='kd_tree').fit(X)
    distances, indices = nbrs.kneighbors(X)
    return indices, distances

def snn_matrix(
    indices: np.ndarray,
    n_neighbors: int
) -> np.ndarray:
    n_samples = indices.shape[0]
    snn = np.zeros((n_samples, n_samples), dtype=np.float32)
    
    for i in range(n_samples):
        for j in range(i + 1, n_samples):
            nni = indices[i]
            nnj = indices[j]
            shared = np.intersect1d(nni, nnj)
            s = [0]
            for shared_knn in shared:
                s.append(n_neighbors - 0.5 * (np.where(nni == shared_knn)[0][0] + np.where(nnj == shared_knn)[0][0]))
            snn[i, j] = max(s)
            snn[j, i] = snn[i, j]
    return snn

In [34]:
def snn_matrix_2(indices: np.ndarray, n_neighbors: int) -> np.ndarray:
    n_samples = indices.shape[0]
    snn = np.zeros((n_samples, n_samples), dtype=np.float32)

    for i in range(n_samples):
        for j in range(i + 1, n_samples):
            nni = indices[i]
            nnj = indices[j]
            shared = np.intersect1d(nni, nnj)

            print(f"Shared neighbors between {i} and {j}: {shared}")  # Debugging

            if len(shared) == 0:  # If no shared neighbors, skip
                continue  

            s = [0]
            for shared_knn in shared:
                s.append(n_neighbors - 0.5 * (np.where(nni == shared_knn)[0][0] + np.where(nnj == shared_knn)[0][0]))

            snn[i, j] = max(s)
            snn[j, i] = snn[i, j]

    return snn


In [4]:
sc_ad=sc.read_h5ad("/Users/anushka/Undergraduate-Project/obsm-anndata-file/sc_data.h5ad")
st_ad=sc.read_h5ad("/Users/anushka/Undergraduate-Project/obsm-anndata-file/st_data.h5ad")
envi_latent_st = st_ad.obsm['envi_latent']
envi_latent_sc = sc_ad.obsm['envi_latent']

print(f"st_ad envi_latent shape: {envi_latent_st.shape}")
print(f"sc_ad envi_latent shape: {envi_latent_sc.shape}")

st_ad envi_latent shape: (18516, 512)
sc_ad envi_latent shape: (7416, 512)


In [7]:
combined_envi_latent = np.concatenate([envi_latent_sc, envi_latent_st], axis=0)
print(combined_envi_latent)

[[ 0.54509723  0.14981279 -0.05097356 ...  0.36242384 -0.03897128
  -0.07992306]
 [ 0.16981411  0.13461766 -0.03672996 ... -0.05966217 -0.00336963
  -0.08696543]
 [ 0.20032202  0.16499552  0.13389239 ...  0.03998933 -0.312753
   0.64815027]
 ...
 [-0.00809197  0.30837688  0.5936734  ... -0.2005235   0.22823815
  -0.25320795]
 [ 0.22977303 -0.25840616 -0.2031456  ... -0.18411052 -0.49793604
  -0.10384995]
 [ 0.18616462  0.04631992  0.4965636  ... -0.09588898  0.2144121
  -0.00381352]]


In [8]:
fit = umap.UMAP(n_neighbors=100, min_dist=0.1, n_components=2)
UMAPEmb = fit.fit_transform(combined_envi_latent)

In [10]:
st_ad.obsm['latent_umap'] = UMAPEmb[:st_ad.shape[0]]
sc_ad.obsm['latent_umap'] = UMAPEmb[st_ad.shape[0]:]

In [15]:
import anndata as ad
cell_types_sc = sc_ad.obs["cell_type"].values
cell_types_st = st_ad.obs["cell_type"].values
combined_cell_types = np.concatenate((cell_types_sc, cell_types_st), axis=0)

adata = ad.AnnData(X=combined_envi_latent)
adata.obs["cell_type"] = combined_cell_types


In [18]:
adata

AnnData object with n_obs × n_vars = 11753 × 512
    obs: 'cell_type', 'n_counts'
    uns: 'log1p'

In [22]:
raw_data = AnnData(adata.X.copy())
raw_data.obs_names, raw_data.var_names = adata.obs_names, adata.var_names
adata.raw = raw_data

sc.pp.normalize_per_cell(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2048)



In [23]:
sc.pp.neighbors(adata, n_neighbors=15)
sc.tl.umap(adata)

In [ ]:
indices, distances = compute_knn_indices(combined_envi_latent, n_neighbors=15)


In [25]:
print("Indices: ", indices)
print("Distances: ", distances)

Indices:  [[    0  3091  6692 ...  4346  3952  4950]
 [    1  7011  6796 ...   442  6850  7306]
 [    2  4440  4090 ...  5286  7014   645]
 ...
 [25929  7832  9233 ...  7284 22317  9464]
 [25930 18605 14278 ...  4075  4654 13134]
 [25931 23511 13914 ...  5098  4309 24276]]
Distances:  [[ 0.          5.47725476  5.50796722 ...  6.02998628  6.03952654
   6.05566414]
 [ 0.          4.95790715  5.1062969  ...  5.42940075  5.47483284
   5.47778131]
 [ 0.          3.95879688  4.57767449 ...  5.18173177  5.2437132
   5.24690745]
 ...
 [ 0.          8.40889324  9.61072271 ... 10.39714577 10.42764172
  10.4283101 ]
 [ 0.         10.59960558 11.31296301 ... 11.668383   11.70930099
  11.76850546]
 [ 0.         10.2162494  10.41680195 ... 11.84110702 11.88605745
  11.89311072]]


In [ ]:
snn_envi = snn_matrix_2(indices, n_neighbors=5)

In [33]:
snn_envi

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)